# Gold Layer - CBBC Stake Distribution
Aggregate silver CBBC stake data into call level ranges for a single trade date and write the result to `gold.cbbc_stake_distribution`.

In [0]:
from pyspark.sql import functions as F

## 1. Set trade date parameter

In [0]:
# Optional local test value
trade_date_str = "2026-01-02"

## 2. Aggregate cash flow by call level range

In [0]:
# Aggregate net cash flow into 50-point call level buckets
query = f"""
WITH cbbc_net AS (
    SELECT
        trade_date,
        cbbc_code,
        bull_bear,
        call_level,
        listing_date,
        SUM(net_cash_flow) AS total_net_cash_flow
    FROM silver.cbbc_stake_curated
    WHERE trade_date = '{trade_date_str}'
    GROUP BY
        trade_date,
        cbbc_code,
        bull_bear,
        call_level,
        listing_date
)
SELECT
    trade_date,
    FLOOR(call_level / 50.0) * 50 AS call_level_range,
    SUM(total_net_cash_flow) AS total_net_cash_flow_within,
    bull_bear
FROM cbbc_net
WHERE call_level BETWEEN 20000 AND 35000
GROUP BY
    trade_date,
    call_level_range,
    bull_bear
ORDER BY call_level_range
"""

df = spark.sql(query)

## 3. Write output to `gold.cbbc_stake_distribution`

In [0]:
# Overwrite the target trade_date partition
(
    df.write
      .format("delta")
      .mode("overwrite")
      .option("replaceWhere", f"trade_date = '{trade_date_str}'")
      .saveAsTable("gold.cbbc_stake_distribution")
)